# 🌐 Global Sanctions Lists Extractor
**By Jahanzeb Khan** · [LinkedIn](https://www.linkedin.com/in/jahanzeb-khan-537756130/)

Runs on Google Colab (16GB RAM — no limits). Downloads all 5 lists, generates Excel, pushes to GitHub Pages.

### Steps
1. Run **Cell 1** — install dependencies
2. Run **Cell 2** — set your GitHub token
3. Run **Cell 3** — upload your `sanctions_extractor.py`
4. Run **Cell 4** — run full extraction
5. Run **Cell 5** — push to GitHub Pages
6. Your download link updates automatically!

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q playwright openpyxl requests
!playwright install chromium
print('✓ All dependencies installed')

In [ ]:
# ── Cell 2: Set your GitHub credentials ───────────────────────────────────────
# Create a GitHub token at: https://github.com/settings/tokens
# Select scope: repo (full control)

GITHUB_TOKEN    = 'YOUR_GITHUB_TOKEN_HERE'   # paste your token
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'      # e.g. jahanzebkhan
GITHUB_REPO     = 'sanctions-extractor-pak'   # your repo name
GITHUB_BRANCH   = 'main'

print(f'✓ GitHub config set: {GITHUB_USERNAME}/{GITHUB_REPO}')

In [ ]:
# ── Cell 3: Upload sanctions_extractor.py ─────────────────────────────────────
from google.colab import files
print('Upload your sanctions_extractor.py file...')
uploaded = files.upload()
print(f'✓ Uploaded: {list(uploaded.keys())}')

In [ ]:
# ── Cell 4: Run full extraction ────────────────────────────────────────────────
import importlib.util, sys
from pathlib import Path
from datetime import datetime

CACHE_DIR   = Path('/content/sanctions_cache')
OUTPUT_FILE = Path('/content/sanctions_output.xlsx')
CACHE_DIR.mkdir(exist_ok=True)

# Load the extractor module
spec = importlib.util.spec_from_file_location('se', '/content/sanctions_extractor.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.CACHE_DIR   = CACHE_DIR
mod.OUTPUT_FILE = OUTPUT_FILE

print('=' * 55)
print('  Global Sanctions Lists Extractor')
print(f'  {datetime.now().strftime("%d %B %Y  %H:%M")}')
print('=' * 55)

# Download all lists
print('\n[1/2] Downloading...\n')
local_files = {}
for key, cfg in mod.SOURCES.items():
    try:
        if key == 'NACTA':
            local_files[key] = mod.download_nacta(cfg)
        else:
            local_files[key] = mod.download(key, cfg)
    except Exception as e:
        print(f'  [{key}] FAILED: {e}')
        local_files[key] = None

# Parse all lists
print('\n[2/2] Parsing...\n')
all_rows, counts = [], {}
for key in mod.ORDER:
    path = local_files.get(key)
    if not path or not Path(path).exists():
        print(f'  [{key}] Skipped')
        counts[key] = 0
        continue
    fmt     = mod.SOURCES[key]['format']
    size_mb = Path(path).stat().st_size / 1024 / 1024
    print(f'  [{key}] {Path(path).name} ({size_mb:.1f} MB) ...', end='', flush=True)
    try:
        rows = mod.PARSERS[fmt](path)
    except Exception as e:
        print(f'\n  ✗ {e}')
        rows = []
    counts[key] = len(rows)
    all_rows.extend(rows)
    print(f'  {len(rows):,} records  ✓')

# Build Excel
mod.build_excel(all_rows, counts)

total = sum(counts.values())
print(f'\n✓ Excel saved: {OUTPUT_FILE}')
print(f'\n  {"Source":<12}  {"Records":>9}')
print(f'  {"-"*12}  {"-"*9}')
for src in mod.ORDER:
    print(f'  {src:<12}  {counts.get(src,0):>9,}')
print(f'  {"TOTAL":<12}  {total:>9,}')

In [ ]:
# ── Cell 5: Push Excel to GitHub Pages ────────────────────────────────────────
import requests, base64, json
from pathlib import Path
from datetime import datetime

def github_push(token, username, repo, branch, filepath, content_bytes, commit_msg):
    """Push a file to GitHub via API."""
    api_url = f'https://api.github.com/repos/{username}/{repo}/contents/{filepath}'
    headers = {
        'Authorization': f'token {token}',
        'Accept': 'application/vnd.github.v3+json',
    }
    # Get current SHA if file exists (needed for update)
    r = requests.get(api_url, headers=headers)
    sha = r.json().get('sha') if r.status_code == 200 else None

    # Push
    payload = {
        'message': commit_msg,
        'content': base64.b64encode(content_bytes).decode(),
        'branch':  branch,
    }
    if sha:
        payload['sha'] = sha

    r = requests.put(api_url, headers=headers, json=payload)
    return r.status_code in (200, 201), r.json()

timestamp = datetime.now().strftime('%d %b %Y %H:%M UTC')
commit_msg = f'Update sanctions data — {timestamp}'

print('Pushing to GitHub...')

# Push Excel file
with open('/content/sanctions_output.xlsx', 'rb') as f:
    excel_bytes = f.read()

ok, resp = github_push(
    GITHUB_TOKEN, GITHUB_USERNAME, GITHUB_REPO, GITHUB_BRANCH,
    'sanctions_output.xlsx', excel_bytes, commit_msg
)
if ok:
    print(f'  ✓ sanctions_output.xlsx pushed ({len(excel_bytes)/1024:.0f} KB)')
else:
    print(f'  ✗ Failed: {resp.get("message","unknown error")}')
    print('    Check your GITHUB_TOKEN has repo scope')

print()
print('✓ Done! Your download link:')
print(f'  https://{GITHUB_USERNAME}.github.io/{GITHUB_REPO}/sanctions_output.xlsx')
print()
print('Your page:')
print(f'  https://{GITHUB_USERNAME}.github.io/{GITHUB_REPO}/')

In [ ]:
# ── Cell 6 (Optional): Download Excel to your computer ────────────────────────
from google.colab import files
files.download('/content/sanctions_output.xlsx')
print('Download started!')